# Etapa 3 · Ensayo controlado con ruido gaussiano

Primero inspecciona **referencia preparada → entrada con ruido gaussiano** sin ejecutar IA. Después, en una celda separada y opcional, ejecuta difusión y compara las tres imágenes. Este notebook reutiliza `src.noise` y `src.experiment` en Python 3.12 con uv; no duplica los cálculos en el kernel.

Usa **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**, si está disponible, para la fase de difusión. La preparación del ruido no necesita GPU. Los cambios deben estar publicados en GitHub antes de preparar una sesión nueva; subir este notebook por sí solo no transfiere los módulos locales.

Se usa la imagen ESA/Hubble `potw1803a.tif`, crédito **NASA, ESA, and J. Comerford (University of Colorado-Boulder)**. Fuente: https://esahubble.org/images/potw1803a/ · CC BY 4.0 según https://esahubble.org/copyright/ · https://creativecommons.org/licenses/by/4.0/. Consulta `assets/ejemplos/README.md` para las atribuciones. La referencia es una imagen publicada ya procesada; no se supone científicamente perfecta ni libre de ruido.

In [ ]:
import os
import signal
import subprocess

def ejecutar_con_salida(comando, cwd=None):
    """Muestra stdout y stderr del proceso en la salida de la celda de VS Code."""
    entorno = os.environ.copy()
    entorno['PYTHONUNBUFFERED'] = '1'
    print('Proceso iniciado. Los mensajes aparecerán debajo de esta celda.', flush=True)
    with subprocess.Popen(
        comando, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1, env=entorno,
        start_new_session=(os.name == 'posix'),
    ) as proceso:
        try:
            for linea in proceso.stdout:
                print(linea, end='', flush=True)
            codigo = proceso.wait()
        except KeyboardInterrupt:
            if proceso.poll() is None:
                if os.name == 'posix':
                    os.killpg(proceso.pid, signal.SIGTERM)
                else:
                    proceso.terminate()
                try:
                    proceso.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    if os.name == 'posix':
                        os.killpg(proceso.pid, signal.SIGKILL)
                    else:
                        proceso.kill()
                    proceso.wait()
            raise
    if codigo != 0:
        raise subprocess.CalledProcessError(codigo, comando)
    print('Proceso finalizado correctamente.', flush=True)


In [ ]:
from pathlib import Path
import subprocess
import sys
import json

if not Path('/content').is_dir():
    raise RuntimeError('Este notebook está preparado para Google Colab.')
proyecto = Path('/content/astro-restauracion')
if not proyecto.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Sabri-Nang/astro-restauracion.git', str(proyecto)], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=proyecto, text=True).strip()
print('Revisión del código:', revision)
if not (proyecto / 'src/experiment.py').is_file():
    raise RuntimeError('Falta src/experiment.py. Publica los cambios y actualiza esta copia con git pull --ff-only origin main antes de continuar.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.3', 'ipywidgets==8.1.7'], check=True)
ejecutar_con_salida(['uv', 'sync', '--locked', '--python', '3.12'], cwd=proyecto)

## Si esta sesión ya tenía una copia del repositorio

Interrumpe cualquier servidor Gradio antes de actualizar. En una celda aparte puedes usar `ejecutar_con_salida(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=proyecto)` y volver a ejecutar la preparación del entorno. Si Git informa cambios locales, revísalos; no los descartes automáticamente.

## Subir el original

Selecciona `potw1803a.tif` sin convertirlo. Se verificará el mismo SHA-256 usado en la prueba técnica anterior. Los originales no se transfieren desde tu PC ni se incluyen al clonar. El selector con ipywidgets sirve también para Colab desde VS Code.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image, Markdown

cargador = widgets.FileUpload(accept='.tif,.tiff', multiple=False, description='Subir TIFF')
display(cargador)

## Preparar e inspeccionar el ruido — sin difusión

Se normaliza RGB a [0, 1], se suma una muestra independiente N(0, sigma²) por píxel y canal, se recorta a [0, 1] y se redondea a 8 bits. El ruido se agrega **después de redimensionar** para que no desaparezca al reducir la imagen. `SIGMA = 0.03` equivale a una desviación de 7,65 niveles en escala 0–255 antes del recorte.

No representa una simulación calibrada del sensor: se opera sobre colores publicados, sin datos crudos ni linealización. El recorte cambia la distribución cerca del negro o del blanco; sus fracciones quedan registradas.

Para comparar niveles, cambia únicamente `SIGMA`, conserva tamaño y semilla y ejecuta de nuevo. Cada combinación utiliza otra carpeta; no se sobrescriben ensayos. Repetir exactamente una combinación exige elegir otra carpeta o reutilizar sus archivos existentes para inspeccionarlos.

In [ ]:
import hashlib

SIGMA = 0.03
SEMILLA_RUIDO = 123
TAMANO = 256
HASH_ESPERADO = '9bc563fef8171c749cc455ad9b0b140652cd26c28187eff4983686bf74f147a2'
entrada = proyecto / 'assets/ejemplos/originales/potw1803a.tif'
entrada.parent.mkdir(parents=True, exist_ok=True)
if entrada.exists() and (not entrada.is_file() or hashlib.sha256(entrada.read_bytes()).hexdigest() != HASH_ESPERADO):
    raise ValueError('Ya existe una entrada diferente en esa ruta; consérvala y elige otra ruta antes de subir el TIFF.')
if not entrada.is_file():
    if not cargador.value:
        raise ValueError('Sube potw1803a.tif y espera a que termine la transferencia.')
    elementos = tuple(cargador.value.values()) if isinstance(cargador.value, dict) else tuple(cargador.value)
    contenido = bytes(elementos[0]['content'])
    if hashlib.sha256(contenido).hexdigest() != HASH_ESPERADO:
        raise ValueError('El archivo no coincide con potw1803a.tif original; no se modificó el original existente.')
    entrada.write_bytes(contenido)

experimento = proyecto / f'outputs/etapa3/gaussiano_sigma{SIGMA:g}_seed{SEMILLA_RUIDO}_size{TAMANO}'
procedencia = {
    'imagen': 'potw1803a.tif',
    'fuente': 'https://esahubble.org/images/potw1803a/',
    'credito': 'NASA, ESA, and J. Comerford (University of Colorado-Boulder)',
    'licencia': 'CC BY 4.0',
    'url_licencia': 'https://creativecommons.org/licenses/by/4.0/',
    'condiciones_fuente': 'https://esahubble.org/copyright/',
    'registro': 'assets/ejemplos/README.md',
}
archivo_procedencia = proyecto / 'outputs/procedencia_gaussiano.json'
archivo_procedencia.parent.mkdir(parents=True, exist_ok=True)
archivo_procedencia.write_text(json.dumps(procedencia, ensure_ascii=False, indent=2), encoding='utf-8')
ejecutar_con_salida(
    ['uv', 'run', '--locked', 'python', '-u', '-m', 'src.experiment', 'prepare',
     '--input', str(entrada), '--output-dir', str(experimento),
     '--sigma', str(SIGMA), '--noise-seed', str(SEMILLA_RUIDO), '--size', str(TAMANO),
     '--provenance', str(archivo_procedencia)], cwd=proyecto,
)
registro = json.loads((experimento / 'experimento.json').read_text(encoding='utf-8'))
registro['revision_proyecto'] = revision
(experimento / 'experimento.json').write_text(json.dumps(registro, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

In [ ]:
display(Markdown('### Referencia preparada / Entrada con ruido gaussiano'))
display(Image(filename=str(experimento / 'comparacion_referencia_ruido.png')))
display(Markdown('### Diferencias referencia–ruido, escala fija 0–255 sin amplificación'))
display(Image(filename=str(experimento / 'diferencias_referencia_ruido.png')))
print(json.dumps(registro['ruido'], ensure_ascii=False, indent=2))
print('Inspecciona: grano del fondo, estrellas tenues, bordes difusos y saturación. Todavía no se ejecutó difusión.')

## Registrar observaciones antes de restaurar

Anota qué cambia entre referencia y entrada ruidosa. El grano sintético no mide el ruido original de la fotografía. Una imagen oscura puede perder muestras negativas por el recorte y presentar un fondo más brillante.

Orden de ensayos posteriores: **Poisson → impulsivo blanco (salt) → desenfoque por convolución**. Aún no están implementados. Poisson necesitará definir una escala de conteos; aplicarlo sobre RGB publicado no reproduce directamente fotones del sensor. Salt permite probar la hipótesis de que puntos blancos pueden convertirse en estructuras parecidas a estrellas; no se asume que suceda. El desenfoque modifica la respuesta espacial y queda como ampliación experimental fuera del MVP; no implica superresolución ni aumento de dimensiones.

## Difusión opcional — ejecutar después de inspeccionar el ruido

En la siguiente celda cambia `EJECUTAR_DIFUSION` a `True` cuando quieras ejecutar esa fase. Se comprueba CUDA antes de cargar el modelo. La configuración inicial usa intensidad 0,2, 30 pasos configurados y semilla del modelo 42: son 6 pasos efectivos con nuestro DDIM. No es un ajuste de eficacia validada. La semilla del ruido permanece separada.

In [ ]:
EJECUTAR_DIFUSION = False
INTENSIDAD = 0.2
PASOS = 30
SEMILLA_MODELO = 42

if not EJECUTAR_DIFUSION:
    print('Difusión omitida. Inspecciona primero el ruido; activa la variable para procesarlo.')
else:
    ejecutar_con_salida(
        ['uv', 'run', '--locked', 'python', '-u', '-c',
         "import sys, torch; assert sys.version_info[:2] == (3, 12); assert torch.cuda.is_available(), 'Selecciona GPU T4 en Colab'; print(torch.cuda.get_device_name(0))"],
        cwd=proyecto,
    )
    salida_difusion = experimento / f'difusion_strength{INTENSIDAD:g}_steps{PASOS}_seed{SEMILLA_MODELO}'
    ejecutar_con_salida(
        ['uv', 'run', '--locked', 'python', '-u', '-m', 'src.experiment', 'restore',
         '--experiment-dir', str(experimento), '--output-dir', str(salida_difusion),
         '--strength', str(INTENSIDAD), '--steps', str(PASOS),
         '--model-seed', str(SEMILLA_MODELO), '--device', 'cuda'], cwd=proyecto,
    )
    display(Markdown('### Referencia preparada / Entrada con ruido / Salida de difusión'))
    display(Image(filename=str(salida_difusion / 'comparacion_triple.png')))
    for nombre, titulo in [
        ('diferencias_ruido_salida.png', 'Cambios del modelo: entrada ruidosa frente a salida'),
        ('diferencias_referencia_salida.png', 'Diferencias respecto de la referencia del ensayo'),
    ]:
        display(Markdown('### ' + titulo))
        display(Image(filename=str(salida_difusion / nombre)))
    print('Reconstruido por IA. Examina pérdidas y posibles artefactos; estos mapas no prueban autenticidad.')

## Descargar y evaluar

La descarga incluye referencia, ruido, mapas, registro de procedencia y parámetros y, si se ejecutó, la salida de difusión. Conserva los créditos al compartir derivadas. No se calcularon PSNR ni SSIM.

Después de difusión registra: ¿se reduce visualmente el grano?, ¿se conservan estrellas tenues?, ¿cambian filamentos, brillos, colores o formas? Señala también resultados desfavorables. No concluyas recuperación de señal astronómica a partir de apariencia visual. Guarda el ZIP antes de finalizar la sesión.

In [ ]:
import shutil
from google.colab import files

if not (experimento / 'experimento.json').is_file():
    raise RuntimeError('Prepara primero el ruido gaussiano.')
archivo_zip = shutil.make_archive(str(Path('/content') / experimento.name), 'zip', experimento)
files.download(archivo_zip)